# 09 — Paper figures and tables

Regenerates **every figure and table in the paper** from result files, in one run.
Nothing here is edited by hand except cell 1, and nothing here recomputes a metric —
if a number is in the paper, it came from a file written by `scripts/evaluate.py`,
`scripts/train.py` (via W&B) or `scripts/preprocess.py`.

**Inputs**

| Source | Produced by |
|---|---|
| `<eval_dir>/per_case_metrics.csv` | `scripts/evaluate.py` on the **test** split |
| `<eval_dir>/summary.csv` | same |
| `<eval_dir>/predictions/<case>.npy` | same — uint8 class map in **ORIGINAL** BraTS geometry |
| `<eval_dir>/probabilities/<case>.npy` | same, if `save_probabilities` was on — float16 `(3, D, H, W)` in **CROPPED** geometry |
| `<eval_dir>/uncertainty/<case>.npy` | *(no producer yet — see §10)* MC-dropout mutual information |
| `<prep_dir>/<case>/{image,label}.npy`, `meta.json` | `scripts/preprocess.py` — **CROPPED** geometry |
| W&B run history | the training sessions |
| `<population_dir>/population_*.{csv,json}` | `scripts/population_stats.py`, ground truth, whole cohort |
| `<report_agreement_dir>/agreement_summary.csv`, `comparison_<a>_vs_<b>.csv` | `scripts/report_agreement.py` |

**Outputs** — `outputs/paper/figures/*.{pdf,png}` and `outputs/paper/tables/*.{md,tex}`.

**Design rules this notebook follows**

1. *Nothing is silently skipped.* Every section that cannot run records why, and §17
   prints the full audit. A figure missing from the paper because a file was absent is
   a bug you find here, not in review.
2. *No metric is recomputed.* Aggregation and formatting only.
3. *Plotting lives in `neurovision.visualization.{figures,tables}`*, which are unit
   tested. This notebook is orchestration: load, call, save, record.

In [1]:
# =========================================================================== #
# THE ONLY CELL TO EDIT
# =========================================================================== #

# One entry per trained model. `eval_dir` is what `inference.evaluation.out_dir`
# was set to for that run. A run whose directory does not exist is reported as
# absent and every section degrades around it -- add rows before the runs finish.
#
# Three OPTIONAL directories per run, each written by its own script and each
# left as None until that script has been run for this model:
#   gates_dir        scripts/extract_gates.py   (explainability.gates.out_dir)
#   attribution_dir  scripts/explain.py         (explainability.attribution.out_dir)
#   calibration_dir  scripts/calibrate.py       (calibration.out_dir)
# They are NOT derived from eval_dir: calibrate.py in particular reads TWO eval
# directories (it fits temperature on val and reports on test), so its output
# belongs to neither one of them.
RUNS = [
    {
        "key": "unet3d",
        "label": "3D U-Net",
        "eval_dir": "../outputs/eval_test",
        "wandb_run_id": "nz5y7li7",
    },
    {
        "key": "swinunetr",
        "label": "SwinUNETR-B",
        "eval_dir": "../outputs/eval_swinunetr_test",
        "wandb_run_id": None,
    },
    {
        "key": "neurovision",
        "label": "NeuroVision-X (ours)",
        "eval_dir": "../outputs/eval_neurovision_test",
        "wandb_run_id": None,
        # Only the full model has fusion gates to extract -- a baseline has no
        # forward_with_gates and scripts/extract_gates.py refuses to run on one.
        "gates_dir": "../outputs/gates_test",
        "attribution_dir": "../outputs/attribution_test",
        "calibration_dir": "../outputs/calibration",
    },
]

# The model the paper argues for. Drives which run picks the qualitative cases
# and which side of every comparison is "A" (so positive improvement = ours better).
OURS_KEY = "neurovision"

# Paired comparisons to run, as (A, B) run keys. A is the model being argued for.
COMPARISONS = [
    ("neurovision", "unet3d"),
    ("neurovision", "swinunetr"),
]

# Ablation ladder (P2 in docs/research/contribution.md). Same dict shape as RUNS.
ABLATION_RUNS = [
    # {"key": "abl_fixed_blend", "label": "Fixed scalar blend", "eval_dir": "../outputs/eval_abl_fixed_test", "wandb_run_id": None},
    # {"key": "abl_content_gate", "label": "Content-only gate", "eval_dir": "../outputs/eval_abl_content_test", "wandb_run_id": None},
]

# W&B. The entity is NOT the username -- it resolves to the default entity, which
# for this account is an institutional one.
WANDB_ENTITY = "amishyadav126-svkm-s-narsee-monjee-institute-of-manageme"
WANDB_PROJECT = "neurovision-x"
USE_WANDB = True  # False skips the training-curve section entirely

PREP_DIR = "../data/preprocessed/brats"
FIGURE_DIR = "../outputs/paper/figures"
TABLE_DIR = "../outputs/paper/tables"

# Phase 5: population anatomy over a whole split (scripts/population_stats.py) and
# report agreement between prediction-derived and ground-truth reports
# (scripts/report_agreement.py). Neither is keyed by RUNS -- each is its own directory,
# not one per model, since there is one population and one agreement study, not one per
# architecture. A path that does not exist skips its section with a recorded reason, the
# same as every optional *_dir above.
POPULATION_DIR = "../outputs/population_gt"
REPORT_AGREEMENT_DIR = "../outputs/report_agreement"

# Qualitative panel. "best_median_worst" ranks by OURS_KEY's dice_mean (falling
# back to the first available run); or give explicit case ids.
QUALITATIVE_POLICY = "best_median_worst"
QUALITATIVE_CASES: list[str] = []
QUALITATIVE_MODALITY = "FLAIR"

# Fusion gate maps (P1). Rows of the gate panel; the gate-vs-boundary profile
# averages over the same cases. Band labels must match metrics.boundary's
# DEFAULT_BANDS, in millimetres from the ground-truth margin -- BraTS is 1 mm
# isotropic, so voxels and millimetres coincide on this data.
GATE_PANEL_CASES = 3
GATE_BANDS = ("0-2", "2-5", "5-10", "10-inf")

# Explainability panel. ATTRIBUTION_REGIONS holds MODEL channel indices
# (0 = ET, 1 = TC, 2 = WT) and must be a subset of what
# explainability.attribution.regions was set to for the run.
ATTRIBUTION_PANEL_CASES = 3
ATTRIBUTION_REGIONS = (0, 2)

# Calibration / risk-coverage are voxel-level and read every saved probability
# map. Cap the case count while iterating; None means the whole split.
CALIBRATION_MAX_CASES = None

SEED = 42  # bootstrap resampling in the statistical comparison
N_BOOT = 10_000

In [2]:
# =========================================================================== #
# Setup -- imports, style, run discovery
# =========================================================================== #
import json
import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from types import SimpleNamespace

from neurovision.analysis.statistics import compare_models, format_comparison
# REGION_NAMES is the MODEL's channel order ("ET", "TC", "WT"); figures.py has
# its own REGION_ORDER ("WT", "TC", "ET") for reporting. They disagree, on
# purpose, and every int -> name mapping below uses this one because it is what
# the scripts write into their CSVs.
from neurovision.data.transforms import REGION_NAMES
from neurovision.inference.postprocess import regions_to_classes
from neurovision.metrics.boundary import distance_band_means, signed_distance_to_boundary
from neurovision.visualization import figures as F
from neurovision.visualization import tables as T

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("matplotlib").setLevel(logging.WARNING)
warnings.filterwarnings("ignore", message=".*constrained_layout.*")

F.use_paper_style()

PREP_DIR = Path(PREP_DIR)
FIGURE_DIR = Path(FIGURE_DIR)
TABLE_DIR = Path(TABLE_DIR)

# --- the audit trail -------------------------------------------------------- #
# Every section appends here. A section that CANNOT run appends a "skipped" row
# with the reason, so a figure absent from the paper is never a silent absence.
AUDIT: list[dict[str, str]] = []


def record(item: str, status: str, detail: str = "") -> None:
    """Log one produced-or-skipped artifact. `status` is 'written' or 'skipped'."""
    AUDIT.append({"item": item, "status": status, "detail": detail})
    marker = "ok  " if status == "written" else "SKIP"
    print(f"[{marker}] {item}" + (f"  -- {detail}" if detail else ""))


def save_fig(fig, stem: str, detail: str = "") -> None:
    paths = F.save_figure(fig, FIGURE_DIR, stem, close=True)
    record(f"figure {stem}", "written", detail or ", ".join(p.name for p in paths))


def save_table(text: str, stem: str, extension: str, detail: str = "") -> None:
    path = T.write_table(text, TABLE_DIR, stem, extension)
    record(f"table {path.name}", "written", detail)


# --- load whatever exists --------------------------------------------------- #
def load_run(run: dict) -> dict | None:
    """Attach the per-case table and available artifact dirs to a run entry.

    Returns None when the run has no evaluation output yet, which is the normal
    state for a run that has not been trained.
    """
    eval_dir = Path(run["eval_dir"])
    per_case_path = eval_dir / "per_case_metrics.csv"
    if not per_case_path.exists():
        return None
    loaded = dict(run)
    loaded["eval_dir"] = eval_dir
    loaded["per_case"] = pd.read_csv(per_case_path, index_col=0)
    loaded["predictions_dir"] = eval_dir / "predictions"
    loaded["probabilities_dir"] = eval_dir / "probabilities"
    loaded["uncertainty_dir"] = eval_dir / "uncertainty"
    # Optional per-script artifact directories. Stored as None unless the
    # directory actually exists, so every section below can test one attribute
    # rather than re-deriving a path and re-checking it.
    for key in ("gates_dir", "attribution_dir", "calibration_dir"):
        raw = run.get(key)
        path = Path(raw) if raw else None
        loaded[key] = path if (path is not None and path.is_dir()) else None
    return loaded


AVAILABLE: dict[str, dict] = {}
for entry in RUNS + ABLATION_RUNS:
    loaded = load_run(entry)
    if loaded is None:
        record(f"run {entry['key']}", "skipped", f"no per_case_metrics.csv under {entry['eval_dir']}")
    else:
        AVAILABLE[entry["key"]] = loaded
        print(f"[ok  ] run {entry['key']:16s} {len(loaded['per_case']):4d} cases  {entry['eval_dir']}")

MAIN_KEYS = [r["key"] for r in RUNS if r["key"] in AVAILABLE]
ABLATION_KEYS = [r["key"] for r in ABLATION_RUNS if r["key"] in AVAILABLE]
LABELS = {k: AVAILABLE[k]["label"] for k in AVAILABLE}

if not AVAILABLE:
    print("\nNo evaluated runs found. Every section below will skip; nothing is broken.")

[ok  ] run unet3d            189 cases  ../outputs/eval_test
[SKIP] run swinunetr  -- no per_case_metrics.csv under ../outputs/eval_swinunetr_test
[SKIP] run neurovision  -- no per_case_metrics.csv under ../outputs/eval_neurovision_test


## 1. Case-set consistency

Every comparison in the paper is **paired** — the same held-out cases, scored by
each model. `compare_models` aligns on `case_id` by index intersection rather
than position, so a mismatch cannot silently compare unrelated cases. But a
large mismatch means two runs were evaluated on different splits, which is a
configuration bug, not a statistical subtlety, and it needs to be visible here.

In [3]:
if len(AVAILABLE) >= 2:
    sets = {k: set(AVAILABLE[k]["per_case"].index) for k in AVAILABLE}
    shared = set.intersection(*sets.values())
    rows = []
    for key, case_ids in sets.items():
        rows.append(
            {
                "run": key,
                "n_cases": len(case_ids),
                "n_shared": len(shared),
                "n_only_here": len(case_ids - shared),
            }
        )
    consistency = pd.DataFrame(rows).set_index("run")
    display(consistency)
    if consistency["n_only_here"].sum() > 0:
        print(
            "\nWARNING: the runs were not evaluated over identical case sets. Paired "
            "comparisons will use the intersection; say so in the paper."
        )
elif AVAILABLE:
    print("Only one run available -- nothing to cross-check yet.")
else:
    record("case-set consistency", "skipped", "no runs available")

Only one run available -- nothing to cross-check yet.


## 2. Training curves

Pulled from the W&B API, not from a local log — a run split across several Kaggle
sessions by resume is ONE W&B run (the id lives in the checkpoint), so there is no
seam to stitch.

Train and validation metrics are logged at different cadences, so the history is
sparse. `plot_training_curves` drops NaNs **per column**; plotting the raw frame
would connect a line straight across the gaps and draw values that were never
measured.

In [4]:
histories: dict[str, pd.DataFrame] = {}

if not USE_WANDB:
    record("figure training_curves", "skipped", "USE_WANDB is False")
else:
    try:
        import wandb

        api = wandb.Api()
        for key in MAIN_KEYS:
            run_id = AVAILABLE[key].get("wandb_run_id")
            if not run_id:
                record(f"training history {key}", "skipped", "no wandb_run_id in the manifest")
                continue
            run = api.run(f"{WANDB_ENTITY}/{WANDB_PROJECT}/{run_id}")
            # samples large enough to defeat W&B's default downsampling, which
            # would visibly smooth the curves.
            histories[LABELS[key]] = run.history(samples=100_000, pandas=True)
            print(f"[ok  ] history {key}: {run.name} ({run.id}) state={run.state}")
    except Exception as exc:  # noqa: BLE001 -- a network/auth failure must not kill the run
        record("training curves", "skipped", f"W&B unavailable: {type(exc).__name__}: {exc}")

if histories:
    panels = [
        F.TrainingPanel("Training loss", ["train/loss_epoch"], "loss"),
        F.TrainingPanel(
            "Validation Dice",
            [f"val/dice_{r}" for r in F.REGION_ORDER] + ["val/dice_mean"],
            "Dice",
            labels=[*F.REGION_ORDER, "mean"],
            ylim=(0.0, 1.0),
        ),
        F.TrainingPanel("Learning rate", ["train/lr"], "LR"),
    ]
    save_fig(F.plot_training_curves(histories, panels), "training_curves",
             f"{len(histories)} run(s)")
elif USE_WANDB:
    record("figure training_curves", "skipped", "no run had a usable W&B history")

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /Users/amish/.netrc.
[ok  ] history unet3d: baseline_unet3d (nz5y7li7) state=finished
[ok  ] figure training_curves  -- 1 run(s)


## 3. Main results table

Mean ± std with the median alongside. The median is load-bearing, not decoration:
per-case Dice here is strongly left-skewed, so the mean alone understates the
typical case while hiding how bad the failures are.

`n_missing` is the count of cases where HD95 was genuinely undefined — one side
of the region empty, the other not. Those are excluded from the mean rather than
given the old BraTS 373.13 mm penalty, so the count travels with the number.

In [5]:
if MAIN_KEYS:
    per_case_tables = {LABELS[k]: AVAILABLE[k]["per_case"] for k in MAIN_KEYS}
    results = T.build_results_table(per_case_tables)
    display(results)

    n_cases = len(next(iter(per_case_tables.values())))
    caption = f"BraTS 2021, {n_cases} held-out test cases. Best per column in bold."
    save_table(T.format_results_markdown(results, caption=caption), "results_main", "md")
    save_table(
        T.format_results_latex(results, caption=caption, label="tab:main-results"),
        "results_main",
        "tex",
    )
else:
    record("table results_main", "skipped", "no evaluated runs")

      model region metric      mean  ...    median    n  n_missing  gt_empty_frac
0  3D U-Net     WT   dice  0.935378  ...  0.957007  189          0       0.000000
1  3D U-Net     WT   hd95  5.356478  ...  2.000000  189          0       0.000000
2  3D U-Net     TC   dice  0.915734  ...  0.963724  189          0       0.000000
3  3D U-Net     TC   hd95  5.209035  ...  1.414214  189          0       0.000000
4  3D U-Net     ET   dice  0.858707  ...  0.928866  189          0       0.026455
5  3D U-Net     ET   hd95  4.017537  ...  1.414214  189          6       0.026455

[6 rows x 9 columns]
[ok  ] table results_main.md
[ok  ] table results_main.tex


## 4. Statistical comparison

Every "A beats B" claim in the paper goes through `compare_models`: paired
bootstrap CI + Wilcoxon signed-rank + effect size, with a Holm correction across
the **whole table**.

Three things worth knowing about the output:

- `improvement` is signed so **positive always means A is better**, whatever the
  metric's own direction. A reader never has to remember the convention per row.
- `verdict = inconclusive` when the CI contains zero **or** Holm-adjusted p > α.
  Those rows must not be claimed.
- The Holm family is one `compare_models` call. Seeing a metric fail and re-running
  on a smaller metric set destroys the error-rate guarantee — so the call is made
  once, over everything, before any p-value is read.

In [6]:
comparison_tables: dict[tuple[str, str], pd.DataFrame] = {}

for key_a, key_b in COMPARISONS:
    if key_a not in AVAILABLE or key_b not in AVAILABLE:
        missing = [k for k in (key_a, key_b) if k not in AVAILABLE]
        record(f"comparison {key_a}_vs_{key_b}", "skipped", f"missing run(s): {missing}")
        continue

    name_a, name_b = LABELS[key_a], LABELS[key_b]
    table = compare_models(
        AVAILABLE[key_a]["per_case"],
        AVAILABLE[key_b]["per_case"],
        generator=np.random.default_rng(SEED),
        name_a=name_a,
        name_b=name_b,
        n_boot=N_BOOT,
    )
    comparison_tables[(key_a, key_b)] = table
    display(table)
    print(format_comparison(table, name_a=name_a, name_b=name_b))

    stem = f"comparison_{key_a}_vs_{key_b}"
    caption = f"{name_a} vs {name_b}, paired over the held-out test cases."
    save_table(
        T.format_comparison_markdown(table, name_a=name_a, name_b=name_b, caption=caption),
        stem,
        "md",
    )
    save_table(
        T.format_comparison_latex(
            table, caption=caption, label=f"tab:{stem}", name_a=name_a, name_b=name_b
        ),
        stem,
        "tex",
    )
    save_fig(
        F.plot_comparison_forest(table, name_a=name_a, name_b=name_b),
        stem,
        f"{len(table)} metrics",
    )

if not COMPARISONS:
    record("statistical comparison", "skipped", "COMPARISONS is empty")

[SKIP] comparison neurovision_vs_unet3d  -- missing run(s): ['neurovision']
[SKIP] comparison neurovision_vs_swinunetr  -- missing run(s): ['neurovision', 'swinunetr']


## 5. Per-case metric distributions

The tail is the result, not the box. Every case is drawn individually, so density
and the failure tail are visible at once — a mean of 0.90 against a median of 0.95
means a minority of cases fail badly enough to move the mean several points, and
for a claim about *reliability* those cases are the point.

In [7]:
if MAIN_KEYS:
    per_case_tables = {LABELS[k]: AVAILABLE[k]["per_case"] for k in MAIN_KEYS}
    save_fig(
        F.plot_metric_distributions(
            per_case_tables, metric="dice", ylabel="Dice", ylim=(-0.02, 1.02)
        ),
        "per_case_dice",
    )
    save_fig(
        F.plot_metric_distributions(per_case_tables, metric="hd95", ylabel="HD95 (mm)"),
        "per_case_hd95",
        "NaN cases (one side empty) are dropped per box",
    )
else:
    record("figure per_case_dice", "skipped", "no evaluated runs")

[ok  ] figure per_case_dice  -- per_case_dice.pdf, per_case_dice.png
[ok  ] figure per_case_hd95  -- NaN cases (one side empty) are dropped per box


## 6. Calibration

The paper's headline is *competitive Dice with substantially better calibration*,
so this section is the claim, not a supplement.

Computed over the **union of predicted-positive and ground-truth-positive voxels**,
never the whole volume: ~99% of voxels are background the model is trivially certain
about, and an unmasked ECE mostly measures tumour size.

Needs `<eval_dir>/probabilities/` — set `inference.evaluation.save_probabilities: true`
before the evaluation run. Probabilities are saved in **cropped** geometry, which is
the frame the model actually predicted in and the frame `label.npy` is already in.

`scripts/calibrate.py` is the fuller, file-driven version of this section: it also
fits a temperature on the **val** split and applies it to **test**, which this
inline computation cannot do (scaling needs raw `logits/`, and fitting on the split
being reported would make the number meaningless). Read
`<calibration_dir>/calibration_metrics.csv` for the temperature-scaled row.


In [8]:
import torch

from neurovision.metrics.segmentation import classes_to_regions
from neurovision.uncertainty.calibration import CalibrationAccumulator, union_foreground_mask

# Channel order of the saved probability maps, fixed by the model's heads.
CHANNEL_REGIONS = ("ET", "TC", "WT")


def load_regions_label(case_id: str) -> np.ndarray | None:
    """Ground truth as a `(3, D, H, W)` binary region mask, in CROPPED geometry."""
    path = PREP_DIR / case_id / "label.npy"
    if not path.exists():
        return None
    classes = torch.from_numpy(np.load(path).astype(np.int64))[None]  # (1, D, H, W)
    return classes_to_regions(classes).numpy().astype(np.float32)


def iter_probability_cases(run: dict):
    """Yield `(case_id, prob, label_regions)` for every case with a saved probability map."""
    directory = run["probabilities_dir"]
    if not directory.exists():
        return
    case_ids = sorted(p.stem for p in directory.glob("*.npy"))
    if CALIBRATION_MAX_CASES is not None:
        case_ids = case_ids[:CALIBRATION_MAX_CASES]
    for case_id in case_ids:
        label = load_regions_label(case_id)
        if label is None:
            continue
        prob = np.load(directory / f"{case_id}.npy").astype(np.float32)
        if prob.shape != label.shape:
            # Cropped-vs-original geometry mismatch: prediction and meta.json
            # came from different preprocessing runs. Loud, never silent.
            print(f"  ! {case_id}: prob {prob.shape} != label {label.shape}; skipped")
            continue
        yield case_id, prob, label


calibration: dict[str, CalibrationAccumulator] = {}
calibration_per_case: dict[str, pd.DataFrame] = {}

for key in MAIN_KEYS:
    run = AVAILABLE[key]
    if not run["probabilities_dir"].exists():
        record(f"calibration {key}", "skipped", "no probabilities/ dir (save_probabilities was off)")
        continue

    accumulator = CalibrationAccumulator(region_names=CHANNEL_REGIONS)
    n_added = 0
    for case_id, prob, label in iter_probability_cases(run):
        mask = union_foreground_mask(torch.from_numpy(prob), torch.from_numpy(label))
        accumulator.add_case(case_id, prob, label, mask=mask)
        n_added += 1
    if n_added == 0:
        record(f"calibration {key}", "skipped", "probabilities/ exists but no case matched a label")
        continue
    calibration[key] = accumulator
    calibration_per_case[key] = accumulator.per_case()
    print(f"[ok  ] calibration {key}: {n_added} cases")
    display(accumulator.summary())

[SKIP] calibration unet3d  -- no probabilities/ dir (save_probabilities was off)


In [9]:
if calibration:
    # One reliability diagram per region. ET is the region the calibration claim
    # leans on, so the regions are never pooled into a single curve.
    for region in F.REGION_ORDER:
        curves = {LABELS[k]: acc.reliability(region) for k, acc in calibration.items()}
        ece = {
            LABELS[k]: float(acc.summary().loc[f"ece_{region}", "value"])
            if f"ece_{region}" in acc.summary().index and "value" in acc.summary().columns
            else float(np.nanmean(calibration_per_case[k][f"ece_{region}"]))
            for k, acc in calibration.items()
        }
        save_fig(
            F.plot_reliability_diagram(curves, ece=ece, title=f"{region} reliability"),
            f"reliability_{region}",
        )

    # Calibration metrics as a results table, reusing the same formatter as Dice
    # so the paper's tables are typographically identical.
    calibration_table = T.build_results_table(
        {LABELS[k]: calibration_per_case[k] for k in calibration},
        metrics=["ece", "brier"],
    )
    display(calibration_table)
    caption = "Per-case calibration on the held-out test split, masked to the tumour union."
    save_table(T.format_results_markdown(calibration_table, caption=caption), "results_calibration", "md")
    save_table(
        T.format_results_latex(
            calibration_table, caption=caption, label="tab:calibration"
        ),
        "results_calibration",
        "tex",
    )
else:
    record("figure reliability_*", "skipped", "no run had usable probability maps")
    record("table results_calibration", "skipped", "no run had usable probability maps")

[SKIP] figure reliability_*  -- no run had usable probability maps
[SKIP] table results_calibration  -- no run had usable probability maps


## 7. Risk–coverage (selective prediction)

"If we refer the least-confident cases to a human, how good is what remains?"

The **oracle** is the ceiling no uncertainty estimate can reach; the **random** line
is the null. A curve hugging the random line means the uncertainty estimate carries
no information about case difficulty — a real, reportable negative result.

**On what the uncertainty scalar is.** With MC-dropout maps present (`<eval_dir>/uncertainty/`)
this uses per-case mutual information, which is *epistemic*. Without them it falls
back to the **predictive entropy of a single deterministic pass**, which is a
different quantity — it contains no epistemic component at all. The two are never
silently interchanged; the fallback is labelled everywhere it appears, and §10 lists
what is needed to produce the real thing.

In [10]:
def bernoulli_entropy(prob: np.ndarray) -> np.ndarray:
    """Per-voxel Bernoulli entropy in nats. Max is ln 2 ~ 0.693, not 1."""
    p = np.clip(prob, 1e-7, 1 - 1e-7)
    return -(p * np.log(p) + (1 - p) * np.log(1 - p))


def per_case_uncertainty(run: dict) -> tuple[pd.Series, str]:
    """Per-case uncertainty scalar, masked to the tumour union.

    Returns the series and a HONEST label describing what the number actually is.
    """
    mc_dir = run["uncertainty_dir"]
    if mc_dir.exists() and any(mc_dir.glob("*.npy")):
        values: dict[str, float] = {}
        for path in sorted(mc_dir.glob("*.npy")):
            case_id = path.stem
            label = load_regions_label(case_id)
            mi = np.load(path).astype(np.float32)
            if label is None or mi.shape != label.shape:
                continue
            mask = label > 0
            values[case_id] = float(mi[mask].mean()) if mask.any() else float("nan")
        return pd.Series(values), "MC-dropout mutual information (epistemic)"

    values = {}
    for case_id, prob, label in iter_probability_cases(run):
        mask = (prob >= 0.5) | (label > 0)
        entropy = bernoulli_entropy(prob)
        values[case_id] = float(entropy[mask].mean()) if mask.any() else float("nan")
    return pd.Series(values), "predictive entropy, single pass (NOT epistemic)"


from neurovision.uncertainty.risk_coverage import (
    oracle_curve,
    random_curve,
    referral_table,
    risk_coverage_curve,
)

rc_curves: dict[str, object] = {}
rc_labels: set[str] = set()
oracle = None
null_curve = None

for key in MAIN_KEYS:
    run = AVAILABLE[key]
    if not run["probabilities_dir"].exists() and not run["uncertainty_dir"].exists():
        record(f"risk-coverage {key}", "skipped", "neither probabilities/ nor uncertainty/ exists")
        continue
    uncertainty, source = per_case_uncertainty(run)
    if uncertainty.empty:
        record(f"risk-coverage {key}", "skipped", "no per-case uncertainty could be computed")
        continue
    rc_labels.add(source)

    scores = run["per_case"]["dice_mean"].reindex(uncertainty.index).dropna()
    uncertainty = uncertainty.reindex(scores.index)
    rc_curves[LABELS[key]] = risk_coverage_curve(uncertainty.to_numpy(), scores.to_numpy())
    if oracle is None:
        oracle = oracle_curve(scores.to_numpy())
        null_curve = random_curve(scores.to_numpy())
    display(referral_table(uncertainty.to_numpy(), scores.to_numpy()))

if rc_curves:
    if len(rc_labels) > 1:
        print(
            "WARNING: models are ranked by DIFFERENT uncertainty sources "
            f"({sorted(rc_labels)}). Their AURCs are not comparable."
        )
    source_note = next(iter(rc_labels))
    save_fig(
        F.plot_risk_coverage(
            rc_curves, oracle=oracle, random=null_curve, title=f"Referral by {source_note}"
        ),
        "risk_coverage",
        source_note,
    )
else:
    record("figure risk_coverage", "skipped", "no run had a usable uncertainty source")

[SKIP] risk-coverage unet3d  -- neither probabilities/ nor uncertainty/ exists
[SKIP] figure risk_coverage  -- no run had a usable uncertainty source


## 8. Qualitative panel

**The geometry trap, handled explicitly.** Saved predictions are in ORIGINAL BraTS
geometry (240×240×155) because that is what a submission requires. The preprocessed
image and label are CROPPED to the case's nonzero bounding box. Overlaying them
directly misaligns by the crop offset and the result *looks entirely plausible*.
Each prediction is therefore re-cropped with the same `bbox` from `meta.json` that
`uncrop_to_original` used, and `QualitativeCase.validate()` raises if the shapes
still disagree.

Each prediction cell draws the prediction filled **and** the ground-truth outline,
so over- and under-segmentation are readable in one cell.

In [11]:
def resolve_uncertainty_source(case_ids: list[str], run: dict) -> tuple[str | None, str]:
    """Pick ONE uncertainty source for the WHOLE panel, or none.

    Resolved once rather than per case on purpose. The panel has a single column
    header, so every row must show the same quantity -- MC-dropout mutual
    information (epistemic) and single-pass predictive entropy are different
    things, and `plot_qualitative_panel` refuses to draw them under one header.
    Resolving per case would mean a panel that renders or raises depending on
    which files happen to exist.
    """
    if all((run["uncertainty_dir"] / f"{c}.npy").exists() for c in case_ids):
        return "mc", "MC-dropout MI"
    if all((run["probabilities_dir"] / f"{c}.npy").exists() for c in case_ids):
        return "entropy", "Entropy (1 pass)"
    return None, "Uncertainty"


def load_qualitative_case(
    case_id: str, model_keys: list[str], source: str | None, source_label: str
) -> F.QualitativeCase | None:
    """Assemble one panel row, everything re-cropped into the preprocessed frame."""
    case_dir = PREP_DIR / case_id
    meta_path = case_dir / "meta.json"
    if not meta_path.exists():
        print(f"  ! {case_id}: no meta.json under {case_dir}; skipped")
        return None
    meta = json.loads(meta_path.read_text())
    image = np.load(case_dir / "image.npy").astype(np.float32)
    ground_truth = np.load(case_dir / "label.npy")

    # The SAME bbox uncrop_to_original used, applied in reverse.
    bbox = [(int(a), int(b)) for a, b in meta["bbox"]]
    crop = tuple(slice(a, b) for a, b in bbox)

    predictions: dict[str, np.ndarray] = {}
    annotations: dict[str, str] = {}
    for key in model_keys:
        run = AVAILABLE[key]
        path = run["predictions_dir"] / f"{case_id}.npy"
        if not path.exists():
            continue
        full = np.load(path)
        if tuple(full.shape) != tuple(meta["original_shape"]):
            raise ValueError(
                f"{case_id}: prediction {full.shape} != original_shape "
                f"{tuple(meta['original_shape'])} -- prediction and meta.json came from "
                "different preprocessing runs."
            )
        predictions[LABELS[key]] = full[crop]
        if case_id in run["per_case"].index:
            annotations[LABELS[key]] = f"Dice {run['per_case'].loc[case_id, 'dice_mean']:.3f}"

    if not predictions:
        print(f"  ! {case_id}: no model has a saved prediction; skipped")
        return None

    # Uncertainty column, honestly labelled -- see section 7. The source was
    # fixed for the whole panel by resolve_uncertainty_source.
    uncertainty = None
    ours = AVAILABLE.get(OURS_KEY) or AVAILABLE[model_keys[0]]
    if source == "mc":
        raw = np.load(ours["uncertainty_dir"] / f"{case_id}.npy").astype(np.float32)
        uncertainty = raw.mean(axis=0) if raw.ndim == 4 else raw
    elif source == "entropy":
        probabilities = np.load(ours["probabilities_dir"] / f"{case_id}.npy").astype(np.float32)
        uncertainty = bernoulli_entropy(probabilities).mean(axis=0)

    return F.QualitativeCase(
        case_id=case_id,
        image=image,
        ground_truth=ground_truth,
        predictions=predictions,
        uncertainty=uncertainty,
        uncertainty_label=source_label,
        annotations=annotations,
    )


def select_cases() -> list[str]:
    if QUALITATIVE_POLICY == "explicit":
        return list(QUALITATIVE_CASES)
    ranking_key = OURS_KEY if OURS_KEY in AVAILABLE else (MAIN_KEYS[0] if MAIN_KEYS else None)
    if ranking_key is None:
        return []
    ranked = AVAILABLE[ranking_key]["per_case"]["dice_mean"].sort_values()
    # worst / median / best, in that reading order.
    return [ranked.index[0], ranked.index[len(ranked) // 2], ranked.index[-1]]


panel_keys = [k for k in MAIN_KEYS if AVAILABLE[k]["predictions_dir"].exists()]
if not panel_keys:
    record("figure qualitative_panel", "skipped", "no run saved predictions/")
else:
    selected = select_cases()
    reference_run = AVAILABLE.get(OURS_KEY) or AVAILABLE[panel_keys[0]]
    source, source_label = resolve_uncertainty_source(selected, reference_run)
    if source is None:
        print("  no uncertainty source covers every selected case; the column is omitted")
    cases = [
        c
        for c in (load_qualitative_case(cid, panel_keys, source, source_label) for cid in selected)
        if c
    ]
    if not cases:
        record("figure qualitative_panel", "skipped", "no case could be assembled")
    else:
        for case in cases:
            print(f"  {case.case_id}: {list(case.predictions)}  unc={source_label}")
        save_fig(
            F.plot_qualitative_panel(cases, modality=QUALITATIVE_MODALITY),
            "qualitative_panel",
            f"{len(cases)} cases x {len(panel_keys)} models",
        )

  no uncertainty source covers every selected case; the column is omitted
  BraTS2021_00412: ['3D U-Net']  unc=Uncertainty
  BraTS2021_00628: ['3D U-Net']  unc=Uncertainty
  BraTS2021_01374: ['3D U-Net']  unc=Uncertainty
[ok  ] figure qualitative_panel  -- 3 cases x 1 models


## 9. Ablation ladder

P2 in `docs/research/contribution.md`: fixed scalar blend → content-only gate →
full gate. The prediction is that the full gate beats the content-only gate on ECE
and HD95 by more than the seed-to-seed std, and ties it on Dice. If it does not,
the contribution is the gate's spatial resolution rather than its input, which is a
smaller and different claim — and must be reported as that.

In [12]:
if ABLATION_KEYS:
    ladder = {LABELS[k]: AVAILABLE[k]["per_case"] for k in ABLATION_KEYS}
    if OURS_KEY in AVAILABLE:
        ladder[LABELS[OURS_KEY]] = AVAILABLE[OURS_KEY]["per_case"]
    ablation = T.build_results_table(ladder)
    display(ablation)
    caption = "Fusion ablation ladder, all parameter-matched, same seed and schedule."
    save_table(T.format_results_markdown(ablation, caption=caption), "results_ablation", "md")
    save_table(
        T.format_results_latex(ablation, caption=caption, label="tab:ablation"),
        "results_ablation",
        "tex",
    )
else:
    record("table results_ablation", "skipped", "ABLATION_RUNS is empty or none evaluated")

[SKIP] table results_ablation  -- ABLATION_RUNS is empty or none evaluated


## 10. Fusion gate maps (P1 — the mechanism fires)

The paper's mechanism evidence: the adaptive gated fusion module is claimed to admit
transformer context **where it is needed**, near tumour margins, rather than being a
decorative extra parameter block.

Two artifacts. The panel shows *where* the gate opens; the profile turns that into a
number — mean gate value against distance to the true boundary, which is what the claim
actually asserts and what a reader can argue with.

Gate maps render at their **true** resolution (nearest-neighbour blocks, not a smooth
upsample): a stride-8 gate is a 12³ map inside a 96³ patch, and smoothing it would present
voxel precision the data does not have. The colour scale is fixed to `[0, 1]` across every
panel, since the gate is a sigmoid initialised at 0.5 — a per-panel min–max would make a
gate that barely moves look dramatic.

In [13]:
# =========================================================================== #
# Fusion gate maps -- the P1 mechanism figure
# =========================================================================== #
# Produced by scripts/extract_gates.py: ONE tumour-centred patch per case, run
# through NeuroVisionX.forward_with_gates in a single pass. Patch-based, not
# sliding-window, because the gates come out as a four-level pyramid at strides
# 2/4/8/16 and MONAI's inferer stitches exactly one output.
#
# Only the full model has fusion blocks, so at most one run contributes here.
gate_runs = {k: AVAILABLE[k]["gates_dir"] for k in MAIN_KEYS if AVAILABLE[k]["gates_dir"]}

if not gate_runs:
    record(
        "figure gate_maps",
        "skipped",
        "no run has a gates_dir -- run scripts/extract_gates.py against a neurovision checkpoint",
    )
    record("figure gate_vs_boundary", "skipped", "no run has a gates_dir")
else:
    gate_key, gate_dir = next(iter(gate_runs.items()))
    manifest_path = gate_dir / "gates_manifest.csv"
    gate_manifest = pd.read_csv(manifest_path, index_col="case_id") if manifest_path.exists() else None

    if gate_manifest is None or gate_manifest.empty:
        record("figure gate_maps", "skipped", f"no gates_manifest.csv under {gate_dir}")
        record("figure gate_vs_boundary", "skipped", f"no gates_manifest.csv under {gate_dir}")
    else:
        # The crop window was chosen by center_on='label' or 'prediction'. A
        # figure caption has to say which -- a window picked using the ground
        # truth is a different claim from one the model chose for itself.
        center_on = set(gate_manifest.get("center_on", pd.Series(dtype=str)).dropna())
        print(f"[info] gate crop centred on: {center_on or 'unrecorded'}")

        gate_case_ids = list(gate_manifest.index[:GATE_PANEL_CASES])
        gate_cases, band_frames = [], {}

        for case_id in gate_case_ids:
            npz_path = gate_dir / f"{case_id}.npz"
            if not npz_path.exists():
                continue
            with np.load(npz_path) as data:
                keys = sorted(
                    (k for k in data.files if k.startswith("gate_level_")),
                    key=lambda k: int(k.rsplit("_", 1)[1]),
                )
                if not keys or "image" not in data.files or "label" not in data.files:
                    continue
                image = data["image"].astype(np.float32)
                # `label` in the .npz is the 3-channel REGION tensor
                # (ET, TC, WT), not a {0,1,2,3} class map. regions_to_classes
                # is the inverse used everywhere else in this project; its
                # outer-to-inner assignment order is load-bearing (reversed, WT
                # paints over every ET voxel and enhancing tumour vanishes), so
                # it is never re-implemented inline.
                regions = torch.from_numpy(data["label"].astype(np.float32)).unsqueeze(0)
                classes = regions_to_classes(regions)[0].numpy().astype(np.uint8)
                gates = {
                    # Level i is the fusion block at stride 2**(i+1) -- the Swin
                    # branch has no stride-1 feature, so the full-resolution CNN
                    # level 0 is never fused and never appears here.
                    f"stride {2 ** (int(k.rsplit('_', 1)[1]) + 1)}": data[k].astype(np.float32)[0]
                    for k in keys
                }

            gate_cases.append(
                F.GateCase(case_id=case_id, image=image, ground_truth=classes, gates=gates)
            )

            # --- gate value as a function of distance to the true margin ---- #
            # This is what turns P1 from three panels into a number: the gate is
            # claimed to open AT the tumour margin, which is a statement about
            # this profile, not about how the heatmap looks.
            wt = (classes > 0).astype(np.uint8)
            sdf = signed_distance_to_boundary(wt, name=f"{case_id} WT")
            for level_name, gate in gates.items():
                # Nearest-neighbour to the label grid: a smooth upsample would
                # imply the coarse gate has voxel precision it does not have.
                factors = [t // s for t, s in zip(sdf.shape, gate.shape)]
                upsampled = gate
                for axis, factor in enumerate(factors):
                    if factor > 1:
                        upsampled = np.repeat(upsampled, factor, axis=axis)
                crop = tuple(slice(0, n) for n in sdf.shape)
                upsampled = upsampled[crop]
                if upsampled.shape != tuple(sdf.shape):
                    continue
                stats = distance_band_means(upsampled, sdf)
                band_frames.setdefault(level_name, []).append(
                    pd.DataFrame(
                        {
                            "band": [b for b in GATE_BANDS],
                            "mean": [stats[f"mean_{b}"] for b in GATE_BANDS],
                            "n": [stats[f"n_{b}"] for b in GATE_BANDS],
                        }
                    )
                )

        if not gate_cases:
            record("figure gate_maps", "skipped", f"no readable .npz under {gate_dir}")
        else:
            save_fig(
                F.plot_gate_maps(gate_cases, modality=QUALITATIVE_MODALITY),
                "gate_maps",
                f"{len(gate_cases)} case(s) from {LABELS[gate_key]}",
            )

        if band_frames:
            # Averaged across cases per level. The gate has no better/worse
            # direction -- it is a mixing coefficient, not an error rate -- so
            # higher_is_better stays None and no arrow is drawn.
            profile = {
                level: pd.concat(frames).groupby("band", sort=False).agg(
                    mean=("mean", "mean"), std=("mean", "std")
                ).reindex(list(GATE_BANDS)).reset_index()
                for level, frames in band_frames.items()
            }
            save_fig(
                F.plot_band_profile(
                    profile,
                    ylabel="Mean gate value",
                    title=f"Transformer context admitted, by distance to margin "
                    f"({len(gate_cases)} cases)",
                ),
                "gate_vs_boundary",
                f"{len(profile)} fusion level(s)",
            )
        else:
            record("figure gate_vs_boundary", "skipped", "no gate level matched the label grid")

[SKIP] figure gate_maps  -- no run has a gates_dir -- run scripts/extract_gates.py against a neurovision checkpoint
[SKIP] figure gate_vs_boundary  -- no run has a gates_dir


## 11. Boundary-stratified error (P3)

HD95 is a single scalar summary of boundary error. This is the same claim in the form that
can be argued with: error rate as a function of distance to the true margin.

`berr = bfnr + bfpr` exactly, so the total is reported alongside its two halves —
under-segmentation (missed tumour) and over-segmentation (spurious tumour) near a margin are
different clinical failures and a pooled rate hides which one a model commits.

Bands are measured from the **ground truth**, so every model is stratified by the same
partition of space. Stratifying each model by its own prediction would compare numbers
computed over different voxel sets.

In [14]:
# =========================================================================== #
# Boundary-stratified error -- the P3 figure and table
# =========================================================================== #
# Error as a FUNCTION of distance to the true tumour margin, which is the
# quantitative form of the boundary-accuracy claim. HD95 is one scalar summary
# of the same thing; this says where the errors actually are.
#
# The columns come from scripts/evaluate.py (inference.evaluation.boundary_bands)
# and are already in the per-case table every other section reads, so nothing
# extra is loaded here.
boundary_per_case = {
    LABELS[k]: AVAILABLE[k]["per_case"]
    for k in MAIN_KEYS
    if any(c.startswith("berr_") for c in AVAILABLE[k]["per_case"].columns)
}

if not boundary_per_case:
    record(
        "figure boundary_error",
        "skipped",
        "no run's per_case_metrics.csv has berr_ columns -- re-run scripts/evaluate.py "
        "(inference.evaluation.boundary_bands is on by default)",
    )
    record("table results_boundary", "skipped", "no run has berr_ columns")
else:
    # berr = bfnr + bfpr exactly, so the two error kinds are reported alongside
    # the total rather than pooled into it. Under-segmentation near a margin
    # and over-segmentation near a margin are different clinical failures.
    for metric, ylabel in (
        ("berr", "Error rate"),
        ("bfnr", "Missed-tumour rate"),
        ("bfpr", "Spurious-tumour rate"),
    ):
        boundary_table = T.build_boundary_table(boundary_per_case, metric=metric)
        for region in F.REGION_ORDER:
            region_rows = boundary_table[boundary_table["region"] == region]
            if region_rows.empty:
                continue
            series = {
                model: region_rows[region_rows["model"] == model].reset_index(drop=True)
                for model in region_rows["model"].unique()
            }
            save_fig(
                F.plot_band_profile(
                    series,
                    ylabel=ylabel,
                    title=f"{region}: {ylabel.lower()} by distance to the true boundary",
                    higher_is_better=False,  # every metric here is an error rate
                ),
                f"boundary_{metric}_{region}",
            )

        if metric == "berr":
            display(boundary_table)
            caption = (
                "Voxel error rate by distance to the ground-truth tumour boundary, "
                "held-out test split."
            )
            save_table(
                T.format_boundary_markdown(boundary_table, caption=caption),
                "results_boundary",
                "md",
            )
            save_table(
                T.format_boundary_latex(
                    boundary_table, caption=caption, label="tab:boundary"
                ),
                "results_boundary",
                "tex",
            )

[SKIP] figure boundary_error  -- no run's per_case_metrics.csv has berr_ columns -- re-run scripts/evaluate.py (inference.evaluation.boundary_bands is on by default)
[SKIP] table results_boundary  -- no run has berr_ columns


## 12. Calibration report from `scripts/calibrate.py`

Section 6 recomputes calibration inline from `probabilities/`, which is valid but cannot do
temperature scaling. This section reads the script's output instead, which can — because the
script fits the temperature on the **validation** split and applies it to **test**.

That split separation is the whole point. A temperature fit on the split it is then reported
on is fit to that split's own noise, and the resulting number would be meaningless. The
script enforces it structurally (two directories, and it raises if they are the same), so
this notebook does not have to.

A `temperature_scaled` variant absent below is not a bug — read `temperature.json`'s
`reason` field.

In [15]:
# =========================================================================== #
# Calibration report produced by scripts/calibrate.py
# =========================================================================== #
# Unlike section 6, which recomputes calibration inline from `probabilities/`,
# this section reads what `scripts/calibrate.py` already wrote. The difference
# that matters is TEMPERATURE SCALING: it needs raw `logits/` (fp16 saturates
# any probability above ~0.99976 to exactly 1.0, whose logit is +inf, so the
# most-confident voxels -- the ones that drive miscalibration -- cannot be
# recovered from a saved probability map), and it must be FIT ON A DIFFERENT
# SPLIT than it is reported on. Neither is something this notebook can do from
# one eval directory, which is why the script exists.
calibration_dirs = {
    k: AVAILABLE[k]["calibration_dir"] for k in MAIN_KEYS if AVAILABLE[k]["calibration_dir"]
}

if not calibration_dirs:
    record(
        "figure calibrate_reliability_*",
        "skipped",
        "no run has a calibration_dir -- run scripts/calibrate.py "
        "calibration.fit_dir=<eval_val> calibration.apply_dir=<eval_test>",
    )
    record("table results_calibrate", "skipped", "no run has a calibration_dir")
    record("figure risk_coverage_calibrate", "skipped", "no run has a calibration_dir")
else:
    # --- the fitted temperature, and whether it may be used at all ---------- #
    temperatures = {}
    for key, cdir in calibration_dirs.items():
        payload_path = cdir / "temperature.json"
        if not payload_path.exists():
            record(f"temperature {key}", "skipped", f"no temperature.json under {cdir}")
            continue
        payload = json.loads(payload_path.read_text())
        temperatures[key] = payload
        if payload.get("converged"):
            print(f"[ok  ] {LABELS[key]:24s} T = {payload['temperature']}")
        else:
            # Printed loudly rather than swallowed: a missing temperature_scaled
            # row below is otherwise an unexplained absence, and the reason is
            # exactly what the paper has to state.
            print(
                f"[SKIP] {LABELS[key]:24s} temperature NOT usable -- "
                f"{payload.get('reason', 'no reason recorded')}"
            )

    # --- reliability, per region, per variant ------------------------------ #
    # Regions are never pooled: ET is the region the calibration claim leans on,
    # and pooling would let WT's much larger voxel count hide ET's behaviour.
    for variant in ("uncalibrated", "temperature_scaled"):
        for region in F.REGION_ORDER:
            curves = {}
            for key, cdir in calibration_dirs.items():
                path = cdir / f"reliability_{variant}_{region}.csv"
                if path.exists():
                    curves[LABELS[key]] = pd.read_csv(path)
            if not curves:
                record(
                    f"figure calibrate_reliability_{variant}_{region}",
                    "skipped",
                    f"no reliability_{variant}_{region}.csv in any calibration_dir"
                    + (
                        " (expected when no temperature converged)"
                        if variant == "temperature_scaled"
                        else ""
                    ),
                )
                continue

            # ECE is read from the script's own pooled summary, never recomputed
            # from the binned means here -- that would be an approximation of an
            # approximation, and the two would disagree in the paper.
            ece = {}
            for key, cdir in calibration_dirs.items():
                metrics_path = cdir / "calibration_metrics.csv"
                if not metrics_path.exists():
                    continue
                metrics = pd.read_csv(metrics_path)
                row = metrics[
                    (metrics["variant"] == variant) & (metrics["metric"] == f"ece_{region}")
                ]
                if len(row) == 1:
                    ece[LABELS[key]] = float(row["value"].iloc[0])

            save_fig(
                F.plot_reliability_diagram(
                    curves,
                    ece=ece or None,
                    title=f"{region} reliability -- {variant.replace('_', ' ')}",
                ),
                f"calibrate_reliability_{variant}_{region}",
            )

    # --- the metrics table, both variants side by side --------------------- #
    metric_frames = []
    for key, cdir in calibration_dirs.items():
        path = cdir / "calibration_metrics.csv"
        if path.exists():
            frame = pd.read_csv(path)
            frame.insert(0, "model", LABELS[key])
            metric_frames.append(frame)

    if metric_frames:
        calibrate_metrics = pd.concat(metric_frames, ignore_index=True)
        display(calibrate_metrics.pivot_table(
            index=["metric"], columns=["model", "variant"], values="value"
        ))
        record(
            "table results_calibrate",
            "written",
            f"{len(calibrate_metrics)} rows across {len(metric_frames)} model(s)",
        )
    else:
        record("table results_calibrate", "skipped", "no calibration_metrics.csv found")

    # --- risk-coverage, with the oracle ceiling and random null ------------ #
    # The null is not garnish: a model curve that hugs it means the uncertainty
    # estimate carries no information about case difficulty, which is a real and
    # reportable negative result. Without it the figure cannot be read.
    rc_curves, rc_oracle, rc_random = {}, None, None
    for key, cdir in calibration_dirs.items():
        rc_path = cdir / "risk_coverage.csv"
        corr_path = cdir / "uncertainty_correlation.json"
        if not rc_path.exists():
            continue
        rc = pd.read_csv(rc_path)
        aurc = json.loads(corr_path.read_text()) if corr_path.exists() else {}
        rc_curves[LABELS[key]] = SimpleNamespace(
            coverage=rc["coverage"].to_numpy(),
            performance=rc["performance"].to_numpy(),
            aurc=float(aurc.get("aurc_model", float("nan"))),
        )
        # One oracle/random pair is enough: they depend only on the score
        # distribution, which is shared across models ONLY if the case set is.
        # Section 1 already asserts that, so the last run's pair is every run's.
        if "oracle_performance" in rc.columns:
            rc_oracle = SimpleNamespace(
                coverage=rc["coverage"].to_numpy(),
                performance=rc["oracle_performance"].to_numpy(),
                aurc=float(aurc.get("aurc_oracle", float("nan"))),
            )
            rc_random = SimpleNamespace(
                coverage=rc["coverage"].to_numpy(),
                performance=rc["random_performance"].to_numpy(),
                aurc=float(aurc.get("aurc_random", float("nan"))),
            )

    if rc_curves:
        save_fig(
            F.plot_risk_coverage(
                rc_curves,
                oracle=rc_oracle,
                random=rc_random,
                title="Selective prediction on the test split",
            ),
            "risk_coverage_calibrate",
        )
        for key, cdir in calibration_dirs.items():
            ref_path = cdir / "referral_table.csv"
            if ref_path.exists():
                display(pd.read_csv(ref_path, index_col=0))
    else:
        record(
            "figure risk_coverage_calibrate",
            "skipped",
            "no risk_coverage.csv -- the evaluation run needs "
            "inference.mc_dropout.enabled=true for per-case uncertainty",
        )

[SKIP] figure calibrate_reliability_*  -- no run has a calibration_dir -- run scripts/calibrate.py calibration.fit_dir=<eval_val> calibration.apply_dir=<eval_test>
[SKIP] table results_calibrate  -- no run has a calibration_dir
[SKIP] figure risk_coverage_calibrate  -- no run has a calibration_dir


## 13. Explainability panel

The headline here is **not** the heatmaps — it is `modality_attribution.csv`.

Enhancing tumour is clinically *defined* by contrast uptake, a T1CE finding. Whole tumour
includes peritumoral oedema, chiefly a FLAIR finding. So Integrated Gradients should show
T1CE dominating for ET and FLAIR for WT. If it does not, the model may be reaching its Dice
through features that are not the radiologically meaningful ones — a reportable finding
about the model, not a bug in the figure.

Integrated Gradients maps are drawn **signed** on a diverging colormap: negative attribution
is evidence *against* the region, which a sequential map would render identically to "not
involved". Grad-CAM with `relu=True` is non-negative and gets the sequential map.

The faithfulness table's `random` row is not garnish. A method that does not beat it on
`insertion_minus_deletion` has not been shown to explain anything.

In [16]:
# =========================================================================== #
# Explainability panel -- what the model actually looked at
# =========================================================================== #
# Produced by scripts/explain.py. The headline is NOT the heatmaps: it is
# modality_attribution.csv. Enhancing tumour is clinically DEFINED by contrast
# uptake (a T1CE finding) and whole tumour includes peritumoral oedema (chiefly
# FLAIR), so Integrated Gradients should show T1CE dominating for ET and FLAIR
# for WT. A model that does not is a reportable finding about the model.
attribution_runs = {
    k: AVAILABLE[k]["attribution_dir"] for k in MAIN_KEYS if AVAILABLE[k]["attribution_dir"]
}

if not attribution_runs:
    record("figure modality_attribution", "skipped", "no run has an attribution_dir")
    record("figure attribution_panel", "skipped", "no run has an attribution_dir")
    record("table faithfulness", "skipped", "no run has an attribution_dir")
else:
    attr_key, attr_dir = next(iter(attribution_runs.items()))

    # --- the radiological sanity check ------------------------------------- #
    modality_path = attr_dir / "modality_attribution.csv"
    if modality_path.exists() and len(pd.read_csv(modality_path)):
        modality_df = pd.read_csv(modality_path)
        # The CSV's `region` column holds the MODEL's channel index
        # (0 = ET, 1 = TC, 2 = WT). figures.py refuses integer regions on
        # purpose -- its own REGION_ORDER is ("WT", "TC", "ET"), so index 0
        # means two different things in the two places and guessing would
        # relabel every bar. Map it here, explicitly, with the model's order.
        modality_df["region"] = modality_df["region"].map(dict(enumerate(REGION_NAMES)))
        save_fig(
            F.plot_modality_attribution(modality_df),
            "modality_attribution",
            f"{modality_df['case_id'].nunique()} case(s) from {LABELS[attr_key]}",
        )
        # Convergence is not decoration: IG's completeness axiom is what makes
        # the attribution mean anything, and a run that exceeded its tolerance
        # must not be read as a scientific result.
        if "delta_exceeded_tolerance" in modality_df.columns:
            n_bad = int(modality_df["delta_exceeded_tolerance"].sum())
            if n_bad:
                print(
                    f"[WARN] {n_bad}/{len(modality_df)} (case, region) rows exceeded IG's "
                    "delta_tolerance -- their attributions are not trustworthy."
                )
    else:
        record("figure modality_attribution", "skipped", f"no modality_attribution.csv in {attr_dir}")

    # --- the qualitative panel --------------------------------------------- #
    attr_manifest_path = attr_dir / "attribution_manifest.csv"
    attr_cases = []
    if attr_manifest_path.exists():
        attr_manifest = pd.read_csv(attr_manifest_path, index_col="case_id")
        for case_id in list(attr_manifest.index[:ATTRIBUTION_PANEL_CASES]):
            npz_path = attr_dir / f"{case_id}.npz"
            if not npz_path.exists():
                continue
            with np.load(npz_path) as data:
                if "image" not in data.files or "label" not in data.files:
                    continue
                image = data["image"].astype(np.float32)
                regions = torch.from_numpy(data["label"].astype(np.float32)).unsqueeze(0)
                classes = regions_to_classes(regions)[0].numpy().astype(np.uint8)
                maps, signed = {}, []
                for region_index in ATTRIBUTION_REGIONS:
                    ig_key = f"ig_region_{region_index}"
                    cam_key = f"cam_region_{region_index}"
                    region = REGION_NAMES[region_index]
                    if ig_key in data.files:
                        # Summed over the modality axis WITHOUT abs, so the map
                        # stays signed: negative values are evidence AGAINST the
                        # region, and a sequential colormap would render "argues
                        # against" and "not involved" identically.
                        maps[f"IG {region}"] = data[ig_key].astype(np.float32).sum(axis=0)
                        signed.append(f"IG {region}")
                    if cam_key in data.files:
                        maps[f"Grad-CAM {region}"] = data[cam_key].astype(np.float32)
                if "attention" in data.files:
                    maps["Attention"] = data["attention"].astype(np.float32)
            if maps:
                attr_cases.append(
                    F.AttributionCase(
                        case_id=case_id, image=image, ground_truth=classes, maps=maps
                    )
                )

    if attr_cases:
        save_fig(
            F.plot_attribution_panel(
                attr_cases,
                modality=QUALITATIVE_MODALITY,
                signed_keys=tuple(k for k in attr_cases[0].maps if k.startswith("IG ")),
            ),
            "attribution_panel",
            f"{len(attr_cases)} case(s)",
        )
    else:
        record("figure attribution_panel", "skipped", f"no readable .npz under {attr_dir}")

    # --- faithfulness, with its null baseline ------------------------------ #
    faith_path = attr_dir / "faithfulness.csv"
    if faith_path.exists() and len(pd.read_csv(faith_path)):
        faith = pd.read_csv(faith_path)
        summary = faith.groupby("method")[
            [c for c in ("deletion_auc", "insertion_auc", "insertion_minus_deletion") if c in faith]
        ].mean()
        display(summary)
        # The `random` row is the whole point: a method that does not beat it on
        # insertion_minus_deletion has not been shown to explain anything.
        if "random" not in summary.index:
            print("[WARN] faithfulness.csv has no `random` row -- the other rows are uninterpretable.")
        record("table faithfulness", "written", f"{len(summary)} method(s) incl. the random null")
    else:
        record("table faithfulness", "skipped", f"no faithfulness.csv in {attr_dir}")

[SKIP] figure modality_attribution  -- no run has an attribution_dir
[SKIP] figure attribution_panel  -- no run has an attribution_dir
[SKIP] table faithfulness  -- no run has an attribution_dir


## 14. Population anatomy (Phase 5)

Cohort-level anatomy from `scripts/population_stats.py`: how often each atlas structure is
involved across a split, how tumour volume distributes over AAL lobes, left/right balance, and
eloquence-involvement rates. Ground truth over all 1,251 cases by default (`POPULATION_DIR`) —
this is not keyed by `RUNS`, because there is one population, not one per architecture.

**Read the eloquence rates as a saturation warning, not a success rate.** Measured over the
full cohort, `frac_near_eloquent` is exactly **1.0**: essentially every glioma in BraTS 2021
directly touches a Sawaya-listed structure, so the field carries no per-case discriminating
information. `population_eloquence.json`'s `degenerate_fields` names every rate that is exactly
0.0 or 1.0 across the cohort — printed loudly below, because a 100% number read without this
context looks like agreement rather than what it is.

In [17]:
# =========================================================================== #
# Population anatomy (Phase 5) -- scripts/population_stats.py
# =========================================================================== #
# One command over a whole split (typically all three frozen splits, ground
# truth): structure-involvement frequency, lobe burden, laterality balance and
# eloquence rates. Unlike every other section in this notebook, this is NOT
# keyed by RUNS -- there is one population, not one per model -- so it reads a
# single directory rather than iterating AVAILABLE.
population_dir = Path(POPULATION_DIR)

if not population_dir.is_dir():
    record(
        "figure population_structure_involvement",
        "skipped",
        f"{population_dir} does not exist -- run scripts/population_stats.py",
    )
    record("figure population_lobe_distribution", "skipped", f"{population_dir} does not exist")
    record("population eloquence rates", "skipped", f"{population_dir} does not exist")
else:
    structures_path = population_dir / "population_structures.csv"
    lobes_path = population_dir / "population_lobes.csv"
    laterality_path = population_dir / "population_laterality.csv"
    eloquence_path = population_dir / "population_eloquence.json"

    if structures_path.exists() and len(pd.read_csv(structures_path)):
        structures = pd.read_csv(structures_path)
        save_fig(
            F.plot_structure_involvement(structures),
            "population_structure_involvement",
            f"{int(structures['n_cases'].iloc[0])} case(s)",
        )
    else:
        record(
            "figure population_structure_involvement",
            "skipped",
            f"no population_structures.csv under {population_dir}",
        )

    if lobes_path.exists() and len(pd.read_csv(lobes_path)):
        lobes = pd.read_csv(lobes_path)
        save_fig(F.plot_lobe_distribution(lobes), "population_lobe_distribution")
    else:
        record(
            "figure population_lobe_distribution",
            "skipped",
            f"no population_lobes.csv under {population_dir}",
        )

    if laterality_path.exists():
        # The population-scale check that would catch a left-right flipped
        # atlas (see CLAUDE.md): a real cohort should show `L` and `R` sharing
        # tumour burden roughly symmetrically. Displayed, not asserted -- this
        # notebook recomputes nothing, it only reads what the script wrote.
        display(pd.read_csv(laterality_path))

    if eloquence_path.exists():
        eloquence = json.loads(eloquence_path.read_text())
        print(f"\nEloquence rates over {eloquence.get('n_cases')} case(s):")
        for key, value in eloquence.items():
            if key in ("n_cases", "degenerate_fields"):
                continue
            print(f"  {key:32s} {value}")

        # Surfaced loudly and first, not buried at the end of the printout: a
        # rate of exactly 1.0 (or 0.0) across the WHOLE cohort is saturation,
        # not agreement, and carries no per-case discriminating information.
        degenerate = eloquence.get("degenerate_fields") or []
        if degenerate:
            print(
                f"\n[WARN] degenerate_fields: {degenerate} -- these rates are exactly 0.0 or "
                "1.0 across the whole cohort. A rate of 1.0 here means essentially every case "
                "touches a Sawaya-listed structure, not that the eloquence layer discriminates "
                "between cases. Do not report this as a success rate."
            )
        record(
            "population eloquence rates",
            "written",
            f"degenerate: {degenerate}" if degenerate else "no degenerate fields",
        )
    else:
        record(
            "population eloquence rates",
            "skipped",
            f"no population_eloquence.json under {population_dir}",
        )


[ok  ] figure population_structure_involvement  -- 1251 case(s)
[ok  ] figure population_lobe_distribution  -- population_lobe_distribution.pdf, population_lobe_distribution.png
   laterality  mean_frac_of_tumour  ...  frac_cases_involved  n_cases
0           L             0.347285  ...             0.769784     1251
1           R             0.304608  ...             0.708233     1251
2     midline             0.002663  ...             0.426059     1251
3  unlabelled             0.345343  ...             1.000000     1251

[4 rows x 6 columns]

Eloquence rates over 1251 case(s):
  frac_any_eloquent                0.9848121502797762
  frac_near_eloquent               1.0
  median_distance_to_eloquent_mm   0.0
  frac_distance_zero               0.988009592326139
  median_n_structures_involved     24.0
  median_frac_unlabelled           0.3411741719547729

[WARN] degenerate_fields: ['frac_near_eloquent'] -- these rates are exactly 0.0 or 1.0 across the whole cohort. A rate of 1.0 here mea

## 15. Report agreement (Phase 5)

Does a more accurate segmentation produce a structured report that agrees more with the report
generated from ground truth? `scripts/report_agreement.py` scores each model's prediction-derived
reports against the ground-truth reports over the held-out test split and runs the same
Holm-corrected paired comparison (`analysis.statistics.compare_models`) used everywhere else in
this notebook — this section reads that output and recomputes nothing.

**The result, stated as it is.** Over 189 paired cases and 25 metrics, Holm-corrected, with patch
size controlled at 64³ across the compared models: `neurovision` vs `baseline_unet3d` has exactly
**one** conclusive improvement (`relerr_vol_TC`), and `capacity_control_unet3d` vs
`baseline_unet3d` has **none**. The +0.0267 ET Dice gain (`docs/experiments.md` note 12) does not
produce a measurably better report. 16 of the report's 25 metrics share an identical median across
every model below, because for a typical case the three reports are the same.

`relerr_*` metrics are ratios with a volume denominator and are heavy-tailed (measured max 128.6
for `relerr_vol_TC`), so their **means** are outlier-driven; the **median** is the number to
read.

In [18]:
# =========================================================================== #
# Report agreement (Phase 5) -- scripts/report_agreement.py
# =========================================================================== #
# Each comparison_<a>_vs_<b>.csv is already an analysis.statistics.compare_models
# table, Holm-corrected over the whole (here, 25-metric) family -- this section
# renders it, it does not recompute anything. Not keyed by RUNS: the file names
# themselves carry the (a, b) pair, since report_agreement runs independently
# of which models happen to be in this notebook's RUNS manifest.
report_agreement_dir = Path(REPORT_AGREEMENT_DIR)

if not report_agreement_dir.is_dir():
    record(
        "figure report_agreement_forest_*",
        "skipped",
        f"{report_agreement_dir} does not exist -- run scripts/report_agreement.py",
    )
    record("table report_agreement_*", "skipped", f"{report_agreement_dir} does not exist")
    record(
        "report_agreement median-agreement count",
        "skipped",
        f"{report_agreement_dir} does not exist",
    )
else:
    comparison_paths = sorted(report_agreement_dir.glob("comparison_*_vs_*.csv"))
    if not comparison_paths:
        record(
            "figure report_agreement_forest_*",
            "skipped",
            f"no comparison_*_vs_*.csv under {report_agreement_dir}",
        )
        record(
            "table report_agreement_*",
            "skipped",
            f"no comparison_*_vs_*.csv under {report_agreement_dir}",
        )
    else:
        for path in comparison_paths:
            pair = path.stem[len("comparison_"):]  # "<a>_vs_<b>"
            if "_vs_" not in pair:
                continue
            name_a, name_b = pair.split("_vs_", 1)
            comparison = pd.read_csv(path, index_col=0)

            n_better = int((comparison["verdict"] == "better").sum())
            n_worse = int((comparison["verdict"] == "worse").sum())
            print(
                f"[info] {name_a} vs {name_b}: {n_better} conclusive improvement(s), "
                f"{n_worse} conclusive regression(s), of {len(comparison)} metric(s)"
            )

            save_fig(
                F.plot_comparison_forest(
                    comparison,
                    name_a=name_a,
                    name_b=name_b,
                    title=f"Report agreement: {name_a} vs {name_b}",
                ),
                f"report_agreement_forest_{name_a}_vs_{name_b}",
            )

            caption = (
                f"Report-agreement metrics, {name_a} vs {name_b}, held-out test split. Positive "
                "improvement means the report generated from this model's prediction agrees more "
                "closely with the report generated from ground truth."
            )
            save_table(
                T.format_comparison_markdown(
                    comparison, name_a=name_a, name_b=name_b, caption=caption
                ),
                f"report_agreement_{name_a}_vs_{name_b}",
                "md",
            )
            save_table(
                T.format_comparison_latex(
                    comparison,
                    name_a=name_a,
                    name_b=name_b,
                    caption=caption,
                    label=f"tab:report_agreement_{name_a}_vs_{name_b}",
                ),
                f"report_agreement_{name_a}_vs_{name_b}",
                "tex",
            )

    # --- the compact statement of the whole result -------------------------- #
    summary_path = report_agreement_dir / "agreement_summary.csv"
    if summary_path.exists() and len(pd.read_csv(summary_path)):
        summary = pd.read_csv(summary_path)
        display(summary.pivot_table(index="metric", columns="model", values="median"))

        n_metrics = int(summary["metric"].nunique())
        n_same_median = int((summary.groupby("metric")["median"].nunique() == 1).sum())
        print(
            f"\n{n_same_median} of {n_metrics} metrics share an identical median across every "
            "model in this summary -- for a typical case, the three reports are the same."
        )
        record(
            "report_agreement median-agreement count",
            "written",
            f"{n_same_median}/{n_metrics} metrics share a median",
        )
    else:
        record(
            "report_agreement median-agreement count",
            "skipped",
            f"no agreement_summary.csv under {report_agreement_dir}",
        )


[info] capacity_control_unet3d vs baseline_unet3d: 0 conclusive improvement(s), 0 conclusive regression(s), of 25 metric(s)
[ok  ] figure report_agreement_forest_capacity_control_unet3d_vs_baseline_unet3d  -- report_agreement_forest_capacity_control_unet3d_vs_baseline_unet3d.pdf, report_agreement_forest_capacity_control_unet3d_vs_baseline_unet3d.png
[ok  ] table report_agreement_capacity_control_unet3d_vs_baseline_unet3d.md
[ok  ] table report_agreement_capacity_control_unet3d_vs_baseline_unet3d.tex
[info] neurovision vs baseline_unet3d: 1 conclusive improvement(s), 0 conclusive regression(s), of 25 metric(s)
[ok  ] figure report_agreement_forest_neurovision_vs_baseline_unet3d  -- report_agreement_forest_neurovision_vs_baseline_unet3d.pdf, report_agreement_forest_neurovision_vs_baseline_unet3d.png
[ok  ] table report_agreement_neurovision_vs_baseline_unet3d.md
[ok  ] table report_agreement_neurovision_vs_baseline_unet3d.tex
[info] neurovision vs capacity_control_unet3d: 1 conclusive im

## 16. Figures this notebook cannot produce

Every producer now exists AND is consumed by a section above. What remains is one
hand-drawn artifact and one deliberate duplication.

| Paper figure | Status |
|---|---|
| Architecture diagram | Hand-drawn. Not a generated artifact and never will be. |
| Inline calibration (§6/§7) vs. `calibrate.py` (§12) | Both are kept on purpose. §6 recomputes calibration from `probabilities/` and works from a single eval directory; §12 reads `scripts/calibrate.py`'s output and is the only one that can report a **temperature-scaled** row, because scaling needs raw `logits/` and a temperature fit on a *different* split. Report §12 when it is available, and say which one the number came from. |

**On MC-dropout.** §6–§8 fall back to the predictive entropy of a single deterministic
pass when no `uncertainty/` directory exists. That fallback is a statement about the RUN,
not about the pipeline — `scripts/evaluate.py` writes real mutual information when
`inference.mc_dropout.enabled=true`. The two are different quantities (a single pass
contains no epistemic component at all), and `resolve_uncertainty_source` labels the
column `Entropy (1 pass)` so they are never presented as the same measurement.

**Every section degrades independently.** A missing directory skips exactly its own
artifacts, with the reason and the command that would produce them, and §17 collects
every skip. A figure absent from the paper should be found there, not in review.


## 17. Audit

The point of the notebook. Every artifact the paper expects is either **written**
here with a path, or **skipped** with a reason. Nothing is silently absent.

In [19]:
audit = pd.DataFrame(AUDIT, columns=["item", "status", "detail"])
display(audit)

written = int((audit["status"] == "written").sum())
skipped = int((audit["status"] == "skipped").sum())
print(f"\n{written} artifact(s) written, {skipped} skipped.")

print(f"\nFigures in {FIGURE_DIR.resolve()}:")
for path in sorted(FIGURE_DIR.glob("*")) if FIGURE_DIR.exists() else []:
    print("  ", path.name)
print(f"\nTables in {TABLE_DIR.resolve()}:")
for path in sorted(TABLE_DIR.glob("*")) if TABLE_DIR.exists() else []:
    print("  ", path.name)

if skipped:
    print(
        "\nEvery skipped row above is a figure or table the paper does NOT have. "
        "Check each one before submission."
    )

                                                 item  ...                                             detail
0                                       run swinunetr  ...  no per_case_metrics.csv under ../outputs/eval_...
1                                     run neurovision  ...  no per_case_metrics.csv under ../outputs/eval_...
2                              figure training_curves  ...                                           1 run(s)
3                               table results_main.md  ...                                                   
4                              table results_main.tex  ...                                                   
5                    comparison neurovision_vs_unet3d  ...                    missing run(s): ['neurovision']
6                 comparison neurovision_vs_swinunetr  ...       missing run(s): ['neurovision', 'swinunetr']
7                                figure per_case_dice  ...               per_case_dice.pdf, per_case_dice.png
8         